In [ ]:
# Uncomment if your platform doesn't already have supporting libraries installed
#!pip install -r requirements.txt > /dev/null

In [ ]:
import torch
import logging
import os
import gc

from supporting_code import load_data, run_experiment, get_dtype, clear_hf_model_cache

In [ ]:
# Load Hugging Face access token
os.environ["HF_TOKEN"] = "REDACTED"

In [ ]:
expanded_dataset = load_data()

In [ ]:
# Grouped the models by size (total parameter count) so we can run
# them on an appropriately-sized GPU, for cost saving. Prices are
# RunPod hourly GPU charges.

# $0.30 (20GB)
model_ids_20gb = [
    "google/gemma-3-270m-it",
    "Qwen/Qwen3.5-0.8B",
    "Qwen/Qwen3.5-2B",
    "google/gemma-2-2b-it",
    "google/gemma-3-4b-it",
    "Qwen/Qwen3.5-4B",
    "google/gemma-4-E2B-it",
]
target_batch_size_20gb = 32

# $0.60 (24GB)
model_ids_24gb = [
	"allenai/Olmo-3-7B-Instruct",
    "allenai/OLMo-2-1124-7B-Instruct",
    "microsoft/Phi-3-small-128k-instruct",
    "Qwen/Qwen2.5-7B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    "meta-llama/Meta-Llama-3-8B-Instruct",
    "Qwen/Qwen3-8B",
    "google/gemma-4-E4B-it",
    "google/gemma-2-9b-it",
    "Qwen/Qwen3.5-9B",
]
target_batch_size_24gb = 64

# $2.20 (96GB)
model_ids_96gb = [
    "google/gemma-4-12B-it",
    "google/gemma-3-12b-it",
    "microsoft/Phi-3-medium-128k-instruct",
    "allenai/OLMo-2-1124-13B-Instruct",
    "Qwen/Qwen3-14B",
    "Qwen/Qwen2.5-14B-Instruct",
    "microsoft/Phi-4-reasoning",
    "microsoft/Phi-4-reasoning-plus",
    "microsoft/phi-4",
    "google/gemma-4-26B-A4B-it",
    "google/gemma-3-27b-it",
    "google/gemma-2-27b-it",
    "Qwen/Qwen3.6-27B",
    "Qwen/Qwen3.5-27B",
    "Qwen/Qwen3-30B-A3B-Instruct-2507",
    "Qwen/Qwen3-30B-A3B",
    "allenai/Olmo-3.1-32B-Instruct",
    "allenai/OLMo-2-0325-32B-Instruct",
    "google/gemma-4-31B-it",
    "Qwen/Qwen2.5-32B-Instruct",
    "Qwen/Qwen3-32B",
    "Qwen/Qwen3.6-35B-A3B",
    "Qwen/Qwen3.5-35B-A3B",
    "microsoft/Phi-3.5-MoE-instruct",
]
target_batch_size_96gb = 64

# $6.50 (180GB B200)
model_ids_180gb = [
    "meta-llama/Llama-3.3-70B-Instruct",
    "meta-llama/Llama-3.1-70B-Instruct",
    "meta-llama/Meta-Llama-3-70B-Instruct",
    "Qwen/Qwen2.5-72B-Instruct",
]
target_batch_size_180gb = 128

# $8.00 (288GB B300)
model_ids_288gb = [
    "meta-llama/Llama-4-Scout-17B-16E-Instruct",
    "Qwen/Qwen3.5-122B-A10B",
]
target_batch_size_288gb = 128


model_ids = model_ids_20gb
target_batch_size = target_batch_size_20gb

In [ ]:
# Run the experiment for all model ids. Start with the target batch
# size but reduce it if we run out of memory.

succeeded = []
failed = []
for model_id in model_ids:
    gc.collect()
    try:
        batch_size = target_batch_size
        while True:
            try:
                run_experiment(model_id, expanded_dataset, get_dtype(), batch_size=batch_size)
                done = True
            except torch.OutOfMemoryError as e:
                if batch_size > 1:
                    batch_size //= 2
                logging.warning(f"OutOfMemoryError. Dropping batch size to {batch_size}.")
                done = False
            if done:
                break
    except Exception as e:
        logging.exception(f"Model {model_id} failed")
        failed.append(model_id)
    else:
        succeeded.append(model_id)
    finally:
        gc.collect()
        torch.cuda.empty_cache()
        clear_hf_model_cache(model_id)

logging.warning(f"Successful experiments: {succeeded}")
logging.warning(f"Failed experiments: {failed}")

In [ ]:
!pip freeze